# ЛР №3: Класифікація - k-NN та Наївний Баєс

**Мета:** Ознайомитися з принципами роботи алгоритмів класифікації - методу k-найближчих сусідів (k-NN) і наївного баєсового класифікатора.

**План:**
1. Застосувати k-NN для вибірок 1-4 з порівнянням sklearn
2. Застосувати наївний Баєс (власна реалізація + sklearn) для вибірок 1-4
3. Порівняти результати двох алгоритмів

## Підготовка середовища

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns

# Налаштування для коректного відображення
import warnings
warnings.filterwarnings('ignore')

# Шляхи
DATA = Path("../Lab1")
FIG = Path("figures")
FIG.mkdir(exist_ok=True)

# Палітра
palette = ["tab:blue", "tab:orange", "tab:green", "tab:red", "tab:purple"]

print("Середовище підготовлено!")

## Завантаження та перевірка даних з Lab1

In [ ]:
# Завантажуємо всі 4 вибірки
sample1 = pd.read_csv(DATA / "sample1_normalized.csv")
sample2 = pd.read_csv(DATA / "sample2_normalized.csv")
sample3 = pd.read_csv(DATA / "sample3_iris_normalized.csv")
sample4 = pd.read_csv(DATA / "sample4_mall_normalized.csv")

# Виводимо структуру кожної вибірки
print("Sample 1:")
print(f"  Shape: {sample1.shape}")
print(f"  Columns: {list(sample1.columns)}")
print()

print("Sample 2:")
print(f"  Shape: {sample2.shape}")
print(f"  Columns: {list(sample2.columns)}")
print()

print("Sample 3 (Iris):")
print(f"  Shape: {sample3.shape}")
print(f"  Columns: {list(sample3.columns)}")
print()

print("Sample 4 (Mall):")
print(f"  Shape: {sample4.shape}")
print(f"  Columns: {list(sample4.columns)}")

## Визначення ознак та міток для кожної вибірки

In [ ]:
# Sample 1: x, y, cluster
X1 = sample1[["x", "y"]].values
y1 = sample1["cluster"].values

# Sample 2: x, y, z, cluster
X2 = sample2[["x", "y", "z"]].values
y2 = sample2["cluster"].values

# Sample 3: sepal length (cm), sepal width (cm), petal length (cm), petal width (cm), species
X3 = sample3[["sepal length (cm)", "sepal width (cm)", "petal length (cm)", "petal width (cm)"]].values
y3 = sample3["species"].values

# Sample 4: Age_norm, Annual_Income_k_norm, Spending_Score_norm, Gender_encoded_norm, Segment
X4 = sample4[["Age_norm", "Annual_Income_k_norm", "Spending_Score_norm", "Gender_encoded_norm"]].values
y4 = sample4["Segment"].values

print("Ознаки та мітки визначено:")
print(f"X1: {X1.shape}, y1: {y1.shape} (унікальні класи: {np.unique(y1)})")
print(f"X2: {X2.shape}, y2: {y2.shape} (унікальні класи: {np.unique(y2)})")
print(f"X3: {X3.shape}, y3: {y3.shape} (унікальні класи: {np.unique(y3)})")
print(f"X4: {X4.shape}, y4: {y4.shape} (унікальні класи: {np.unique(y4)})")

## Розділення на навчальну та тестову вибірки (70/30)

In [ ]:
# Розділяємо кожну вибірку
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.3, random_state=42, stratify=y1)
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.3, random_state=42, stratify=y2)
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.3, random_state=42, stratify=y3)
X4_train, X4_test, y4_train, y4_test = train_test_split(X4, y4, test_size=0.3, random_state=42, stratify=y4)

print("Розділення на train/test (70/30):")
print(f"Sample 1: train={X1_train.shape}, test={X1_test.shape}")
print(f"Sample 2: train={X2_train.shape}, test={X2_test.shape}")
print(f"Sample 3: train={X3_train.shape}, test={X3_test.shape}")
print(f"Sample 4: train={X4_train.shape}, test={X4_test.shape}")

---
# Завдання 1: k-NN (k-Nearest Neighbors)

## 1.1. Функція для тестування k-NN з різними k

In [ ]:
def test_knn_with_multiple_k(X_train, X_test, y_train, y_test, k_values):
    """Тестує k-NN з різними значеннями k."""
    results = []
    for k in k_values:
        knn = KNeighborsClassifier(n_neighbors=k)
        knn.fit(X_train, y_train)
        y_pred = knn.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        results.append({'k': k, 'accuracy': acc})
        print(f"  k={k:2d}: accuracy={acc:.4f}")
    return pd.DataFrame(results)

print("Функція готова!")

## 1.2. Тестування k-NN на всіх вибірках

In [ ]:
k_values = [1, 3, 5, 7, 9, 11, 15, 21]

print("Sample 1 - k-NN:")
results_knn_1 = test_knn_with_multiple_k(X1_train, X1_test, y1_train, y1_test, k_values)

print("\nSample 2 - k-NN:")
results_knn_2 = test_knn_with_multiple_k(X2_train, X2_test, y2_train, y2_test, k_values)

print("\nSample 3 (Iris) - k-NN:")
results_knn_3 = test_knn_with_multiple_k(X3_train, X3_test, y3_train, y3_test, k_values)

print("\nSample 4 (Mall) - k-NN:")
results_knn_4 = test_knn_with_multiple_k(X4_train, X4_test, y4_train, y4_test, k_values)

## 1.3. Графіки залежності accuracy від k

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

all_results = [
    (results_knn_1, "Sample 1: Намальовані кластери"),
    (results_knn_2, "Sample 2: Згенеровані дані"),
    (results_knn_3, "Sample 3: Iris Dataset"),
    (results_knn_4, "Sample 4: Mall Customers")
]

for idx, (results, title) in enumerate(all_results):
    ax = axes[idx]
    ax.plot(results['k'], results['accuracy'], marker='o', linewidth=2, markersize=8, color='steelblue')
    ax.set_xlabel('k (кількість сусідів)', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(k_values)
    
    # Найкраще k
    best_idx = results['accuracy'].idxmax()
    best_k = results.loc[best_idx, 'k']
    best_acc = results.loc[best_idx, 'accuracy']
    ax.axvline(best_k, color='red', linestyle='--', alpha=0.5, label=f'Best k={int(best_k)}')
    ax.legend()

plt.tight_layout()
plt.savefig(FIG / "knn_accuracy_vs_k.png", dpi=120, bbox_inches='tight')
plt.show()

print("\nНайкращі значення k:")
for idx, (results, title) in enumerate(all_results):
    best_idx = results['accuracy'].idxmax()
    best_k = int(results.loc[best_idx, 'k'])
    best_acc = results.loc[best_idx, 'accuracy']
    print(f"{title}: k={best_k}, accuracy={best_acc:.4f}")

## 1.4. Приклад класифікації окремих елементів

In [ ]:
# Беремо найкраще k для Sample 1
best_k = int(results_knn_1.loc[results_knn_1['accuracy'].idxmax(), 'k'])
knn_best = KNeighborsClassifier(n_neighbors=best_k)
knn_best.fit(X1_train, y1_train)

# Класифікуємо перші 10 тестових точок
n_examples = min(10, len(X1_test))
y_pred_examples = knn_best.predict(X1_test[:n_examples])

print(f"Приклад класифікації для Sample 1 (k={best_k}):")
print("="*70)
for i in range(n_examples):
    true_label = y1_test[i]
    pred_label = y_pred_examples[i]
    match = "OK" if true_label == pred_label else "ERROR"
    print(f"Point {i+1}: x={X1_test[i][0]:.3f}, y={X1_test[i][1]:.3f} | True: {true_label} | Pred: {pred_label} [{match}]")

---
# Завдання 2: Наївний Баєсів Класифікатор

## 2.1. Власна реалізація Gaussian Naive Bayes

In [ ]:
class CustomGaussianNB:
    """Власна реалізація Gaussian Naive Bayes."""
    
    def __init__(self):
        self.classes = None
        self.class_priors = {}  # P(C)
        self.means = {}         # μ для кожної ознаки в кожному класі
        self.vars = {}          # σ² для кожної ознаки в кожному класі
    
    def fit(self, X, y):
        """Навчання моделі."""
        self.classes = np.unique(y)
        n_samples = len(y)
        
        for c in self.classes:
            X_c = X[y == c]
            
            # Апріорна ймовірність P(C)
            self.class_priors[c] = len(X_c) / n_samples
            
            # Середнє та дисперсія для кожної ознаки
            self.means[c] = np.mean(X_c, axis=0)
            self.vars[c] = np.var(X_c, axis=0) + 1e-9  # додаємо epsilon для стабільності
        
        return self
    
    def _gaussian_pdf(self, x, mean, var):
        """Обчислює гаусівську ймовірність."""
        eps = 1e-9
        coef = 1.0 / np.sqrt(2 * np.pi * var + eps)
        exponent = np.exp(-((x - mean) ** 2) / (2 * var + eps))
        return coef * exponent
    
    def _predict_single(self, x):
        """Передбачає клас для одного зразка."""
        posteriors = {}
        
        for c in self.classes:
            # Використовуємо log для стабільності
            posterior = np.log(self.class_priors[c])
            
            for i in range(len(x)):
                prob = self._gaussian_pdf(x[i], self.means[c][i], self.vars[c][i])
                posterior += np.log(prob + 1e-9)
            
            posteriors[c] = posterior
        
        return max(posteriors, key=posteriors.get)
    
    def predict(self, X):
        """Передбачає класи для всіх зразків."""
        return np.array([self._predict_single(x) for x in X])

print("Власна реалізація Gaussian Naive Bayes готова!")

## 2.2. Тестування обох реалізацій Naive Bayes

In [ ]:
def test_naive_bayes(X_train, X_test, y_train, y_test, sample_name):
    """Тестує власну та sklearn реалізації Naive Bayes."""
    print(f"\n{sample_name}:")
    print("="*60)
    
    # Власна реалізація
    custom_nb = CustomGaussianNB()
    custom_nb.fit(X_train, y_train)
    y_pred_custom = custom_nb.predict(X_test)
    acc_custom = accuracy_score(y_test, y_pred_custom)
    
    # Sklearn
    sklearn_nb = GaussianNB()
    sklearn_nb.fit(X_train, y_train)
    y_pred_sklearn = sklearn_nb.predict(X_test)
    acc_sklearn = accuracy_score(y_test, y_pred_sklearn)
    
    print(f"Власна реалізація: accuracy={acc_custom:.4f}")
    print(f"Sklearn GaussianNB: accuracy={acc_sklearn:.4f}")
    print(f"Різниця: {abs(acc_custom - acc_sklearn):.4f}")
    
    return {
        'sample': sample_name,
        'custom_accuracy': acc_custom,
        'sklearn_accuracy': acc_sklearn,
        'custom_predictions': y_pred_custom,
        'sklearn_predictions': y_pred_sklearn
    }

# Тестуємо на всіх вибірках
results_nb_1 = test_naive_bayes(X1_train, X1_test, y1_train, y1_test, "Sample 1")
results_nb_2 = test_naive_bayes(X2_train, X2_test, y2_train, y2_test, "Sample 2")
results_nb_3 = test_naive_bayes(X3_train, X3_test, y3_train, y3_test, "Sample 3 (Iris)")
results_nb_4 = test_naive_bayes(X4_train, X4_test, y4_train, y4_test, "Sample 4 (Mall)")

all_nb_results = [results_nb_1, results_nb_2, results_nb_3, results_nb_4]

## 2.3. Порівняння власної та sklearn реалізацій

In [ ]:
sample_names = ['Sample 1', 'Sample 2', 'Sample 3\n(Iris)', 'Sample 4\n(Mall)']
custom_accs = [r['custom_accuracy'] for r in all_nb_results]
sklearn_accs = [r['sklearn_accuracy'] for r in all_nb_results]

x = np.arange(len(sample_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width/2, custom_accs, width, label='Власна реалізація', color='steelblue')
bars2 = ax.bar(x + width/2, sklearn_accs, width, label='Sklearn GaussianNB', color='coral')

ax.set_xlabel('Вибірка', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Порівняння реалізацій Naive Bayes', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sample_names)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

# Додаємо значення
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIG / "naive_bayes_comparison.png", dpi=120, bbox_inches='tight')
plt.show()

## 2.4. Приклад класифікації для Sample 3 (Iris)

In [ ]:
# Навчаємо на Sample 3
nb_iris = CustomGaussianNB()
nb_iris.fit(X3_train, y3_train)

# Класифікуємо 10 тестових зразків
n_examples = min(10, len(X3_test))
y_pred_iris = nb_iris.predict(X3_test[:n_examples])

print("Приклад класифікації для Sample 3 (Iris) - Naive Bayes:")
print("="*80)
feature_names = ["sepal_len", "sepal_wid", "petal_len", "petal_wid"]

for i in range(n_examples):
    true_label = y3_test[i]
    pred_label = y_pred_iris[i]
    match = "OK" if true_label == pred_label else "ERROR"
    
    features_str = ", ".join([f"{name}={X3_test[i][j]:.2f}" for j, name in enumerate(feature_names)])
    print(f"Sample {i+1}: [{features_str}]")
    print(f"  True: {true_label} | Predicted: {pred_label} [{match}]")
    print()

## 2.5. Матриця помилок для Sample 3

In [ ]:
cm = confusion_matrix(y3_test, results_nb_3['custom_predictions'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.xlabel('Передбачений клас', fontsize=12)
plt.ylabel('Справжній клас', fontsize=12)
plt.title('Матриця помилок - Sample 3 (Iris)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG / "confusion_matrix_iris.png", dpi=120, bbox_inches='tight')
plt.show()

print("\nЗвіт класифікації:")
print(classification_report(y3_test, results_nb_3['custom_predictions']))

---
# Завдання 3: Порівняння k-NN та Naive Bayes

## 3.1. Зведена таблиця результатів

In [ ]:
# Збираємо найкращі результати k-NN
best_knn = [
    {'sample': 'Sample 1', 'k': int(results_knn_1.loc[results_knn_1['accuracy'].idxmax(), 'k']), 
     'acc': results_knn_1['accuracy'].max()},
    {'sample': 'Sample 2', 'k': int(results_knn_2.loc[results_knn_2['accuracy'].idxmax(), 'k']), 
     'acc': results_knn_2['accuracy'].max()},
    {'sample': 'Sample 3', 'k': int(results_knn_3.loc[results_knn_3['accuracy'].idxmax(), 'k']), 
     'acc': results_knn_3['accuracy'].max()},
    {'sample': 'Sample 4', 'k': int(results_knn_4.loc[results_knn_4['accuracy'].idxmax(), 'k']), 
     'acc': results_knn_4['accuracy'].max()}
]

# Створюємо таблицю
comparison_data = []
for i in range(4):
    comparison_data.append({
        'Вибірка': best_knn[i]['sample'],
        'k-NN (k)': best_knn[i]['k'],
        'k-NN Accuracy': f"{best_knn[i]['acc']:.4f}",
        'Naive Bayes Accuracy': f"{all_nb_results[i]['custom_accuracy']:.4f}",
        'Різниця': f"{abs(best_knn[i]['acc'] - all_nb_results[i]['custom_accuracy']):.4f}",
        'Кращий': 'k-NN' if best_knn[i]['acc'] > all_nb_results[i]['custom_accuracy'] else 'Naive Bayes'
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nЗведена таблиця порівняння:")
print("="*100)
print(comparison_df.to_string(index=False))

comparison_df.to_csv("comparison_results.csv", index=False, encoding='utf-8-sig')
print("\nТаблицю збережено у comparison_results.csv")

## 3.2. Візуальне порівняння k-NN та Naive Bayes

In [ ]:
knn_accs = [r['acc'] for r in best_knn]
nb_accs = [r['custom_accuracy'] for r in all_nb_results]

x = np.arange(4)
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, knn_accs, width, label='k-NN (найкраще k)', color='mediumseagreen')
bars2 = ax.bar(x + width/2, nb_accs, width, label='Naive Bayes', color='mediumpurple')

ax.set_xlabel('Вибірка', fontsize=13)
ax.set_ylabel('Accuracy', fontsize=13)
ax.set_title('Порівняння k-NN та Naive Bayes', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sample_names)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

# Додаємо значення
for i, bar in enumerate(bars1):
    height = bar.get_height()
    k_val = best_knn[i]['k']
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}\n(k={k_val})', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIG / "knn_vs_naive_bayes.png", dpi=120, bbox_inches='tight')
plt.show()

## 3.3. Аналіз результатів

In [ ]:
print("\n" + "="*100)
print("АНАЛІЗ РЕЗУЛЬТАТІВ")
print("="*100)

knn_wins = sum(1 for i in range(4) if knn_accs[i] > nb_accs[i])
nb_wins = sum(1 for i in range(4) if nb_accs[i] > knn_accs[i])
ties = 4 - knn_wins - nb_wins

print(f"\nПеремоги:")
print(f"  k-NN: {knn_wins} разів")
print(f"  Naive Bayes: {nb_wins} разів")
if ties > 0:
    print(f"  Нічия: {ties} разів")

avg_knn = np.mean(knn_accs)
avg_nb = np.mean(nb_accs)
print(f"\nСередня точність:")
print(f"  k-NN: {avg_knn:.4f}")
print(f"  Naive Bayes: {avg_nb:.4f}")

print(f"\n{'='*100}")
print("ВИСНОВКИ:")
print(f"{'='*100}")

for i in range(4):
    sample = ['Sample 1', 'Sample 2', 'Sample 3 (Iris)', 'Sample 4 (Mall)'][i]
    diff = abs(knn_accs[i] - nb_accs[i])
    winner = "k-NN" if knn_accs[i] > nb_accs[i] else "Naive Bayes"
    
    if diff < 0.01:
        print(f"\n{sample}: Практично однакові результати (різниця {diff:.4f})")
    else:
        print(f"\n{sample}: {winner} кращий на {diff:.4f}")
        if winner == "k-NN":
            print(f"  Оптимальне k={best_knn[i]['k']}, точність {knn_accs[i]:.4f}")
        else:
            print(f"  Naive Bayes досяг точності {nb_accs[i]:.4f}")

print(f"\n{'='*100}")
print("Загальний висновок:")
if avg_knn > avg_nb:
    print(f"k-NN показав кращі результати (+{avg_knn - avg_nb:.4f} accuracy)")
elif avg_nb > avg_knn:
    print(f"Naive Bayes показав кращі результати (+{avg_nb - avg_knn:.4f} accuracy)")
else:
    print("Обидва алгоритми показали однакові результати")
print(f"{'='*100}")

---
# Підсумки

У цій лабораторній роботі ми:

1. **Застосували k-NN** до 4 вибірок з різними значеннями k та побудували графіки залежності точності від k
2. **Реалізували власний Gaussian Naive Bayes** та порівняли його з sklearn (результати практично ідентичні)
3. **Порівняли k-NN та Naive Bayes** між собою для всіх вибірок

Всі результати збережено:
- `figures/` - графіки
- `comparison_results.csv` - зведена таблиця